# Human Activity Recognition (HAR): Hyperparameter Tuning

This notebook reproduces the tuning of the LdaBoost-based approach

In [ ]:
# Reproducible setup and imports
import os
import random
import logging
from typing import Dict

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score 

# Global seed
RANDOM_SEED: int = 42
os.environ["PYTHONHASHSEED"] = str(RANDOM_SEED)
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
logging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s")

# Import LdaBoost from local package
import sys
REL_PROJECT_ROOT = "../../"
if REL_PROJECT_ROOT not in sys.path:
    sys.path.insert(0, REL_PROJECT_ROOT)
from LdaBoost import LdaBoost


## Dataset and preprocessing


In [3]:
import re

def clean_column_names(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df.columns = [re.sub(r"[^A-Za-z0-9_]+", "_", c) for c in df.columns]
    return df

train_df = pd.read_csv("Data/train.csv")
test_df = pd.read_csv("Data/test.csv", names=list(train_df.columns), header=0)

train_df = clean_column_names(train_df)
test_df = clean_column_names(test_df)

X = train_df.drop(columns="Activity")
y = train_df["Activity"]
X_test = test_df.drop(columns="Activity")
y_test = test_df["Activity"]

le = LabelEncoder()
y_enc = le.fit_transform(y)
y_test_enc = le.transform(y_test)

X_np = X.to_numpy()



## Grid search on train_df


In [4]:
X_tr, X_val, y_tr, y_val = train_test_split(
    X_np, y_enc, test_size=0.2, random_state=RANDOM_SEED, stratify=y_enc
)

param_grid = {
    "n_estimators": [100, 200, 300],
    "learning_rate": [0.05, 0.1, 0.2],
    "max_depth": [2, 3, 4],
}

best_params = None
best_score = -1.0
total = len(param_grid["n_estimators"]) * len(param_grid["learning_rate"]) * len(param_grid["max_depth"])
i = 0
for n in param_grid["n_estimators"]:
    for lr in param_grid["learning_rate"]:
        for md in param_grid["max_depth"]:
            i += 1
            params = {
                "n_estimators": n,
                "learning_rate": lr,
                "max_depth": md,
                "random_state": RANDOM_SEED,
            }
            model = LdaBoost(**params)
            model.fit(X_tr, y_tr)
            y_pred = model.predict(X_val)
            score = accuracy_score(y_val, y_pred)
            print(f"Finished iteration {i}/{total}: {params} -> val_acc={score:.4f}")
            if score > best_score:
                best_score = score
                best_params = params

best_params, best_score


Finished iteration 1/27: {'n_estimators': 100, 'learning_rate': 0.05, 'max_depth': 2, 'random_state': 42} -> val_acc=0.9660
Finished iteration 2/27: {'n_estimators': 100, 'learning_rate': 0.05, 'max_depth': 3, 'random_state': 42} -> val_acc=0.9742
Finished iteration 3/27: {'n_estimators': 100, 'learning_rate': 0.05, 'max_depth': 4, 'random_state': 42} -> val_acc=0.9762
Finished iteration 4/27: {'n_estimators': 100, 'learning_rate': 0.1, 'max_depth': 2, 'random_state': 42} -> val_acc=0.9748
Finished iteration 5/27: {'n_estimators': 100, 'learning_rate': 0.1, 'max_depth': 3, 'random_state': 42} -> val_acc=0.9748
Finished iteration 6/27: {'n_estimators': 100, 'learning_rate': 0.1, 'max_depth': 4, 'random_state': 42} -> val_acc=0.9762
Finished iteration 7/27: {'n_estimators': 100, 'learning_rate': 0.2, 'max_depth': 2, 'random_state': 42} -> val_acc=0.9742
Finished iteration 8/27: {'n_estimators': 100, 'learning_rate': 0.2, 'max_depth': 3, 'random_state': 42} -> val_acc=0.9755
Finished iter

({'n_estimators': 200,
  'learning_rate': 0.1,
  'max_depth': 3,
  'random_state': 42},
 0.9768864717878993)

## Train final LdaBoost and evaluate on test set


In [5]:
final_model = LdaBoost(**best_params)
final_model.fit(X_np, y_enc)
y_test_pred = final_model.predict(X_test.to_numpy())
test_acc = accuracy_score(y_test_enc, y_test_pred)
print(f"Test accuracy: {test_acc:.4f}")


Test accuracy: 0.9508
